In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Início rápido do Executor

<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Semelhante à primitiva [Sampler](/guides/get-started-with-sampler), o Executor amostra registradores de saída de execuções de Circuit quântico, mas não tem supressão ou mitigação de erros integradas. Em vez disso, ele faz parte do [modelo de execução direcionada](/guides/directed-execution-model) que fornece os ingredientes para capturar intenções de design no lado do cliente e transfere a geração custosa de variantes de Circuit para o lado do servidor. O Executor segue as diretivas fornecidas em anotações de Circuit e opções, gera e vincula valores de parâmetros, executa os Circuits vinculados no hardware e retorna os resultados de execução e metadados. Ele não toma decisões implícitas por você e oferece controle total e transparência.

> **Note:** O pacote Qiskit ainda não tem uma classe base para a primitiva Executor.

## Antes de começar
Alguns dos exemplos de código nesta página usam `samplex`, que faz parte do pacote Samplomatic. Portanto, antes de executar esses blocos de código, você deve instalar o Samplomatic, conforme mostrado no bloco de código a seguir. Para mais informações, veja a [documentação do Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Criar e transpilar um Circuit
Você precisa de pelo menos um Circuit para usar a primitiva Executor. Ele pode opcionalmente ter parâmetros.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

O Circuit precisa ser transformado para usar apenas instruções suportadas pela QPU (referido como circuito de *arquitetura de conjunto de instruções (ISA)*). Use o Transpiler para isso.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inicializar um `QuantumProgram`
Inicialize um `QuantumProgram` com sua carga de trabalho. Um `QuantumProgram` é composto de `QuantumProgramItems`. Normalmente, cada item consiste em um Circuit, um conjunto de valores de parâmetros e possivelmente um `samplex` para randomizar o conteúdo do Circuit. Para detalhes completos, veja [Entradas e saídas do Executor](/guides/executor-input-output).

A célula a seguir inicializa um `QuantumProgram` e especifica a realização de 25 shots. Em seguida, ele anexa o Circuit alvo transpilado.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opcional: Agrupar gates e medições em caixas anotadas
Agrupar instruções em caixas e anotá-las é a principal forma de especificar sua intenção. No exemplo a seguir, usamos `generate_boxing_pass_manager` e seus parâmetros de twirling para agrupar gates de dois Qubits e medições em caixas e aplicar anotação de twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Invocar o Executor e obter resultados
Execute o `QuantumProgram` em um Backend IBM&reg; usando a primitiva `Executor` com opções padrão. Veja [Opções do Executor](/guides/executor-options) para saber sobre as opções disponíveis.